# Лекција 11 - Протокол агент-агент (A2A)


## Подешавање


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Шта је A2A протокол?

**Agent-to-Agent (A2A) протокол** је отворени стандард који омогућава AI агентима да комуницирају,
откривају једни друге, и сарађују — чак и када су изграђени на различитим оквирима или хостирани
од стране различитих сервиса.

Кључни појмови:

- **Откривање** – Агенти објављују *Картицу агента* која описује њихове способности, чинећи то
  лакшим за друге агенте (или оркестраторе) да пронађу правог специјалисту за задатак.
- **Прослеђивање порука** – Агенти размењују структурисане поруке путем заједничког протокола, тако да
  захтев од једног агента може бити разумљен и испуњен од стране другог без обзира на његову унутрашњу
  имплементацију.
- **Животни циклус задатка** – A2A дефинише статусе као што су *предато*, *у раду*, *завршено*, и
  *неуспешно*, дајући оркестратору потпуну видљивост у томе како напредује делегирани задатак.

У овој лекцији симулирамо сарадњу у A2A стилу повезивањем три специјализована туристичка агента
у радни ток где сваки агент доприноси својом стручношћу и прослеђује резултате следећем.


## Креирање специјализованих туристичких агената


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Сарадња више агената путем радног тока

Повезујемо три агента у секвенцијални ток рада који одражава A2A размену порука:

1. **CurrencyExchangeAgent** прима кориснички захтев и пружа смернице у вези са валутом.
2. **ActivityPlannerAgent** прима обогаћени контекст и додаје препоруке за активности.
3. **TravelManagerAgent** синтетизује оба уноса у финални путни брифинг.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Разумевање A2A у продукционом окружењу

У продукционом окружењу A2A протокол откључава моћне сценарије преко сервиса:

| Могућност | Опис |
|---|---|
| **Интероперабилност између фрејмворка** | Агент изграђен у једном фрејмворку може делегирати задатке агенту изграђеном у било ком другом фрејмворку који је у складу са A2A, омогућавајући право међуорганизационо умрежавање. |
| **Границе сервиса** | Агенти могу живети у одвојеним микросервисима, регионима у облаку или чак у различитим организацијама, а ипак беспрекорно сарађивати. |
| **Динамичко откривање** | Оркестратор може упитати регистар картица агената у време извршавања да пронађе најприкладнијег стручњака за дати подзадатак. |
| **Стриминг & push обавештења** | A2A подржава Server-Sent Events (SSE) за ажурирања напретка у реалном времену и push обавештења за дуготрајне задатке. |

Радни ток који смо горе направили је поједностављена, унутарпроцесна верзија овог обрасца. У стварном
размештању сваки агент би изложио HTTP крајњу тачку, објавио картицу агента и комуницирао
преко A2A JSON-RPC протокола.


## Резиме

У овој лекцији сте научили:

1. **Шта је A2A протокол** — отворени стандард за откривање агената један према другом, размену порука и управљање задацима.
2. **Како креирати специјализоване агенте** — агент за размену валута, агент за планирање активности и оркестратор за управљање путовањима.
3. **Како повезати агенте у ток рада** — користећи `WorkflowBuilder` за моделовање секвенцијалног прослеђивања порука између агената.
4. **Како A2A функционише у продукцији** — омогућавајући сарадњу између различитих фрејмворка и сервиса са динамичким откривањем и ажурирањима у реалном времену.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Изјава о одрицању одговорности:
Овај документ је преведен помоћу услуге за превод засноване на вештачкој интелигенцији [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да превод буде тачан, имајте у виду да аутоматизовани преводи могу да садрже грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетом. За критичне информације препоручује се професионалан људски превод. Не сносимо одговорност за било какве неспоразуме или погрешна тумачења која проистекну из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
